# Stochastic PAC-Bayesian Transformers: Reproducibility Demo

**Paper:** *Stochastic PAC-Bayesian Transformers for Network Intrusion Detection and Natural Language Processing Applications*  
**Authors:** Roger Nick Anaedevha, Alexander G. Trofimov, Yuri V. Borodachev  
**Affiliation:** National Research Nuclear University MEPhI, Moscow, Russia

This notebook demonstrates the core components of the framework:
1. Model architecture (Bayesian attention, variational embeddings)
2. Multi-objective loss function
3. Adversarial attacks (FGSM, PGD, C&W with EOT)
4. Uncertainty quantification (epistemic vs. aleatoric)
5. Calibration evaluation (ECE, MCE, Brier, AUROC)
6. Active learning with uncertainty-guided acquisition

---

## Datasets

**Network Intrusion Detection:**
- Integrated IDPS Security 3Datasets (CIC-IoT-2023, CSE-CICIDS2018, UNSW-NB15)
- Available at: https://doi.org/10.34740/KAGGLE/DSV/12479689

**Toxic Content Detection:**
- MetaHate, HatEval, Founta (standard academic repositories)

**Fake News Detection:**
- LIAR, FakeNewsNet, ISOT (standard academic repositories)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from src.models.stochastic_transformer import StochasticPACBayesianTransformer, ModelConfig
from src.models.bayesian_attention import BayesianMultiHeadAttention, BayesianLinear
from src.attacks.fgsm import FGSM
from src.attacks.pgd import PGD
from src.attacks.eot import EOTWrapper
from src.training.losses import MultiObjectiveLoss
from src.evaluation.calibration import CalibrationMetrics
from src.evaluation.uncertainty import UncertaintyQuantifier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Model Architecture

The stochastic transformer uses **Bayesian attention** at layers {3, 6, 9, 12} of a 12-layer encoder.  
Variational weights W ~ N(mu, sigma^2) are sampled via the reparameterization trick.

**Key hyperparameters** (Table F.1 of supplementary):
- 12 layers, 8 heads, d_model=256, d_ff=1024
- Prior: N(W_BERT, 0.01*I)
- Posterior init: log_var = -6.0 (var ~ 0.0025)
- MC samples: 30 (train), 50 (inference)

In [ ]:
# --- Tabular model (Network IDS) ---
config_net = ModelConfig(
    num_layers=12, num_heads=8, d_model=256, d_ff=1024,
    bayesian_layers=[3, 6, 9, 12],
    input_dim=40, num_classes=8, input_type='tabular',
    mc_train_samples=30, mc_inference_samples=50,
)
model_net = StochasticPACBayesianTransformer(config_net).to(device)

total = sum(p.numel() for p in model_net.parameters())
trainable = sum(p.numel() for p in model_net.parameters() if p.requires_grad)
print(f'Network IDS model: {total:,} total params, {trainable:,} trainable')

# Test forward pass with synthetic data
x_dummy = torch.randn(4, 40, device=device)
logits = model_net(x_dummy)
print(f'Logits shape: {logits.shape}')  # (4, 8)

In [ ]:
# --- KL divergence computation ---
kl = model_net.compute_kl_loss()
print(f'KL divergence (initial): {kl.item():.4f}')

# Bayesian layers contribute KL; deterministic layers do not
for i, layer in enumerate(model_net.layers, 1):
    kl_i = layer.kl_divergence().item()
    tag = ' [Bayesian]' if layer.bayesian else ''
    if kl_i > 0:
        print(f'  Layer {i}{tag}: KL = {kl_i:.4f}')

## 2. Multi-Objective Loss (Eq. 9)

L_tot = L_cls + 0.01*L_KL + 0.05*L_cal + 0.2*L_adv + 0.001*L_reg

Selected via two-stage grid search (Appendix F).

In [ ]:
criterion = MultiObjectiveLoss(
    lambda_kl=0.01, lambda_cal=0.05, lambda_adv=0.2, lambda_reg=0.001)

# Simulate clean + adversarial forward pass
targets = torch.randint(0, 8, (4,), device=device)
logits_clean = model_net(x_dummy)
logits_adv = model_net(x_dummy + 0.05 * torch.randn_like(x_dummy))

losses = criterion(logits_clean, targets, model_net, adv_logits=logits_adv)
for k, v in losses.items():
    print(f'{k:>8s}: {v.item():.6f}')

## 3. Adversarial Attacks with EOT

EOT adversaries average gradients over M=5 stochastic model samples,
requiring perturbations to succeed across multiple weight instantiations.

In [ ]:
# FGSM with EOT
fgsm = FGSM(epsilon=0.1)
eot_fgsm = EOTWrapper(fgsm, eot_samples=5)

x_adv_fgsm = eot_fgsm.generate(model_net, x_dummy, targets)
perturbation = (x_adv_fgsm - x_dummy).abs().max().item()
print(f'FGSM+EOT perturbation (L-inf): {perturbation:.4f}')

# PGD with EOT
pgd = PGD(epsilon=0.1, steps=10, step_size=0.01)
eot_pgd = EOTWrapper(pgd, eot_samples=5)

x_adv_pgd = eot_pgd.generate(model_net, x_dummy, targets)
perturbation_pgd = (x_adv_pgd - x_dummy).abs().max().item()
print(f'PGD+EOT perturbation (L-inf): {perturbation_pgd:.4f}')

## 4. Monte Carlo Uncertainty Decomposition

- **Epistemic**: variance across MC samples (model uncertainty)
- **Aleatoric**: mean per-sample entropy (data uncertainty)
- **Total**: entropy of the mean predictive distribution

In [ ]:
uq = UncertaintyQuantifier(model_net, num_samples=50, device=device)
result = uq.predict_with_uncertainty(x_dummy)

print('Predictions:', result['predictions'].cpu().numpy())
print(f'Epistemic uncertainty:  {result["epistemic"].cpu().numpy()}')
print(f'Aleatoric uncertainty:  {result["aleatoric"].cpu().numpy()}')
print(f'Total uncertainty:      {result["total"].cpu().numpy()}')

## 5. Calibration Metrics

Paper results (Table 3): ECE=0.043, MCE=0.101, Brier=0.061, AUROC=0.892

In [ ]:
cal = CalibrationMetrics(n_bins=15)

# Simulate predictions for demonstration
np.random.seed(42)
n = 1000
num_classes = 8
true_labels = np.random.randint(0, num_classes, n)
# Well-calibrated predictions (close to paper ECE = 0.043)
logits_sim = np.random.randn(n, num_classes) * 2.0
logits_sim[np.arange(n), true_labels] += 3.0
probs_sim = np.exp(logits_sim) / np.exp(logits_sim).sum(axis=1, keepdims=True)

metrics = cal.compute_all(probs_sim, true_labels)
for k, v in metrics.items():
    print(f'{k:>8s}: {v:.4f}')

In [ ]:
# --- Reliability diagram ---
def reliability_diagram(probs, targets, n_bins=15):
    confidences = probs.max(axis=-1)
    predictions = probs.argmax(axis=-1)
    correct = (predictions == targets).astype(float)
    
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() > 0:
            bin_accs.append(correct[mask].mean())
            bin_confs.append(confidences[mask].mean())
            bin_counts.append(mask.sum())
    
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.bar(bin_confs, bin_accs, width=1/n_bins, alpha=0.6, edgecolor='k')
    ax.set_xlabel('Predicted Confidence')
    ax.set_ylabel('Empirical Accuracy')
    ax.set_title(f'Reliability Diagram (ECE={metrics["ece"]:.3f})')
    ax.legend()
    plt.tight_layout()
    plt.show()

reliability_diagram(probs_sim, true_labels)

## 6. Training on Real Data

To train on real datasets, download the data and run:

```bash
# Network intrusion detection
python scripts/train.py \
    --domain network \
    --dataset cic_iot_2023 \
    --data_path data/cic_iot_2023.csv \
    --config configs/default_config.yaml

# Toxic content detection
python scripts/train.py \
    --domain toxic \
    --dataset metahate \
    --data_path data/metahate.csv \
    --config configs/default_config.yaml

# Fake news detection
python scripts/train.py \
    --domain fakenews \
    --dataset liar \
    --data_path data/liar.csv \
    --config configs/default_config.yaml
```

Run the full experimental suite across all 9 datasets, 5 folds, 3 seeds:

```bash
bash scripts/run_experiments.sh /path/to/data /path/to/output
```

## Paper Results Summary

| Domain | Accuracy | F1 | ECE | Retention |
|--------|----------|-----|------|----------|
| Network Intrusion | 96.8 +/- 0.7 | 96.5 +/- 0.8 | 0.042 +/- 0.005 | 88.7% |
| Toxic Content | 96.2 +/- 0.9 | 95.8 +/- 1.0 | 0.049 +/- 0.007 | 87.6% |
| Fake News | 97.1 +/- 0.7 | 96.8 +/- 0.8 | 0.037 +/- 0.006 | 87.8% |
| **Average** | **96.8 +/- 0.8** | **96.4 +/- 0.9** | **0.043 +/- 0.006** | **88.3%** |

Active learning reduces labeling requirements by **68%** (35% labels -> 95% of full-data performance).